# Source Code Scouting

In [1]:
!pip install -q datasets

import json
from datasets import load_dataset

ds = load_dataset("LLM4Binary/decompile-eval", split="train", streaming=True)

# In thử 1 mẫu để xác nhận tên cột và giá trị thực tế
first_sample = next(iter(ds))
print("Các cột:", list(first_sample.keys()))
print("opt:", repr(first_sample.get("opt")), "| language:", repr(first_sample.get("language")))

targets = {"O0": 250, "O1": 250, "O2": 250, "O3": 250}
collected = {k: [] for k in targets}

for row in ds:
    opt = str(row.get("opt", "")).strip()
    lang = str(row.get("language", "")).strip().lower()
    
    if lang == "c" and opt in targets:
        if len(collected[opt]) < targets[opt]:
            # Đảm bảo có cột 'pseudo' nếu script normalize yêu cầu mặc định
            if "pseudo" not in row and "ida_pseudo" in row:
                row["pseudo"] = row["ida_pseudo"]
            collected[opt].append(row)
            
    if all(len(collected[k]) == targets[k] for k in targets):
        break

raw_subset = []
for opt in ["O0", "O1", "O2", "O3"]:
    raw_subset.extend(collected[opt])
    print(f"Đã lấy {opt}: {len(collected[opt])} mẫu")

assert len(raw_subset) > 0, "LỖI: Không lọc được mẫu nào!"

with open("/kaggle/working/raw_subset_1000.json", "w", encoding="utf-8") as f:
    json.dump(raw_subset, f, ensure_ascii=False, indent=2)

!ls -lh /kaggle/working/raw_subset_1000.json

README.md:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

Các cột: ['index', 'func_name', 'func_dep', 'func', 'test', 'opt', 'language', 'asm', 'ida_asm', 'ida_pseudo', 'ghidra_asm', 'ghidra_pseudo']
opt: 'O0' | language: 'c'
Đã lấy O0: 250 mẫu
Đã lấy O1: 250 mẫu
Đã lấy O2: 250 mẫu
Đã lấy O3: 250 mẫu
-rw-r--r-- 1 root root 10M Nov 20 07:13 /kaggle/working/raw_subset_1000.json


In [2]:
SCRIPT_PATH = "/kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos/LLM4Decompile/sk2decompile/evaluation/normalize_pseudo.py"

# 1. In ra nội dung script normalize_pseudo.py để xem nó đang đọc key nào và dùng tree-sitter ra sao
!cat -n {SCRIPT_PATH} | head -n 120

     1	import re
     2	import json
     3	import argparse
     4	from multiprocessing import Pool, cpu_count
     5	from tqdm import tqdm  # ✅ 添加进度条模块
     6	import random
     7	
     8	import subprocess
     9	
    10	def good_func(func):
    11	    func = '{'.join(func.split('{')[1:])
    12	    func_sp = func.split('\n')
    13	    total = 0
    14	    for line in func_sp:
    15	        if len(line.strip())>=3:
    16	            total+=1
    17	    if total>3 and total<300:
    18	        return True
    19	    return False
    20	def strip_empty(code):
    21	    return "\n".join(line for line in code.splitlines() if line.strip())
    22	def format_with_clang(func: str, style: str = "Google") -> str:
    23	    # Build the command
    24	    if not func:
    25	        return None
    26	    cmd = ["clang-format", f"--style={style}"]
    27	    try:
    28	        proc = subprocess.run(
    29	            cmd,
    30	            input=func,
    31	            text=True,
    32	

In [3]:
import json
with open("/kaggle/working/raw_subset_1000.json", "r", encoding="utf-8") as f:
    data = json.load(f)
print("Số mẫu:", len(data))
print("Các key hiện có:", list(data[0].keys()))
for k, v in data[0].items():
    print(f" - {k}: {str(v)[:80]}...")

Số mẫu: 1000
Các key hiện có: ['index', 'func_name', 'func_dep', 'func', 'test', 'opt', 'language', 'asm', 'ida_asm', 'ida_pseudo', 'ghidra_asm', 'ghidra_pseudo', 'pseudo']
 - index: 0...
 - func_name: mi_cmp_dynamic_unique...
 - func_dep: eloqsql/storage/myisam/mi_dynrec.c...
 - func: int _mi_cmp_dynamic_unique(MI_INFO *info, MI_UNIQUEDEF *def,
			   const uchar *...
 - test: ...
 - opt: O0...
 - language: c...
 - asm: mi_cmp_dynamic_unique:
pushq %rbp
movq %rsp, %rbp
subq $0x50, %rsp
movq %rdi, -0...
 - ida_asm: _mi_cmp_dynamic_unique:
push    rbp
mov     rbp, rsp
sub     rsp, 50h
mov     [r...
 - ida_pseudo: long long  mi_cmp_dynamic_unique(char *a1, long long a2, long long a3, const cha...
 - ghidra_asm: _mi_cmp_dynamic_unique:
PUSH RBP
MOV RBP,RSP
SUB RSP,0x50
MOV qword ptr [RBP + -...
 - ghidra_pseudo: 
int _mi_cmp_dynamic_unique(long *param_1,long param_2,int8 param_3,int8 param_4...
 - pseudo: long long  mi_cmp_dynamic_unique(char *a1, long long a2, long long a3, const cha...


## Fixed Data

In [4]:
# Cài đặt thư viện Python và clang-format (bắt buộc cho normalize_pseudo.py)
!pip install -q tree-sitter==0.24.0 tree-sitter-c==0.23.4 tqdm
!apt-get update -qq && apt-get install -y -qq clang-format
!clang-format --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libllvm18:amd64.
(Reading database ... 125687 files and directories currently installed.)
Preparing to unpack .../libllvm18_1%3a18.1.3-1ubuntu1_amd64.deb ...
Unpacking libllvm18:amd64 (1:18.1.3-1ubuntu1) ...
Selecting previously unselected package libclang-cpp18.
Preparing to unpack .../libclang-cpp18_1%3a18.1.3-1ubuntu1_amd64.deb ...
Unpacking libclang-cpp18 (1:18.1.3-1ubuntu1) ...
Selecting previously unselected package clang-format-18.
Preparing to unpack .../clang-format-18_1%3a18.1.3-1ubuntu1_amd64.deb ...
Unpacking clang-format-18 (1:18.1.3-1ubuntu1) ...
Selecting previously unselected package clang-format:amd64.
Preparing to unpack .../clang-format_1%3a18.0-59~exp2_amd64.deb ...
Unpacking clang-format:amd64 (1:18.0-59~exp2) ...
Setting up libllvm18:amd64

In [5]:
SCRIPT_PATH = "/kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos/LLM4Decompile/sk2decompile/evaluation/normalize_pseudo.py"

# 1. Xem nội dung/tham số của script để biết chính xác tên flag
!python {SCRIPT_PATH} --help 

usage: normalize_pseudo.py [-h] [--input_json INPUT_JSON]
                           [--output_json OUTPUT_JSON] [--key_name KEY_NAME]
                           [--workers WORKERS] [--remove REMOVE]

并行处理 IDA 伪代码字符串列表

options:
  -h, --help            show this help message and exit
  --input_json INPUT_JSON
                        输入 JSON 文件路径（每项为字符串）
  --output_json OUTPUT_JSON
                        输出 JSON 文件路径
  --key_name KEY_NAME   输出 JSON 文件路径
  --workers WORKERS     进程数默认使用8核心
  --remove REMOVE       remove fail cases


In [6]:
import json

SCRIPT_PATH = "/kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos/LLM4Decompile/sk2decompile/evaluation/normalize_pseudo.py"

# 1. Chạy normalize và xuất ra file .json tạm
!python {SCRIPT_PATH} \
    --input_json /kaggle/working/raw_subset_1000.json \
    --output_json /kaggle/working/normalized_subset_1000.json \
    --key_name ida_pseudo

# 2. Chuyển mảng JSON sang đúng chuẩn JSONL (mỗi dòng 1 object)
with open("/kaggle/working/normalized_subset_1000.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Tổng số mẫu sau normalize:", len(data))
assert len(data) > 0, "LỖI: File vẫn rỗng!"

with open("/kaggle/working/decompile-eval-subset-1000.jsonl", "w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

# 3. Kiểm tra file đầu ra
!ls -lh /kaggle/working/decompile-eval-subset-1000.jsonl
!head -n 1 /kaggle/working/decompile-eval-subset-1000.jsonl | cut -c 1-200

[+] 开始处理 1000 条记录，使用 32 个进程
Processing: 100%|███████████████████████████| 1000/1000 [00:12<00:00, 77.73it/s]
[✓] 完成处理：/kaggle/working/raw_subset_1000.json:1000 → /kaggle/working/normalized_subset_1000.json:880
Tổng số mẫu sau normalize: 880
-rw-r--r-- 1 root root 9.3M Nov 20 07:14 /kaggle/working/decompile-eval-subset-1000.jsonl
{"index": 0, "func_name": "mi_cmp_dynamic_unique", "func_dep": "eloqsql/storage/myisam/mi_dynrec.c", "func": "int _mi_cmp_dynamic_unique(MI_INFO *info, MI_UNIQUEDEF *def,\n\t\t\t   const uchar *record
